In [ ]:
import torch
from ultralytics import YOLO
from torch_operation_counter import OperationsCounterMode
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import numpy as np
import time
warnings.filterwarnings('ignore')

In [ ]:
model_sizes = ['nano', 'medium', 'large']
dim_min, dim_max = 32, 3200
dims = range(dim_min, dim_max+1, 32)

In [ ]:
dim = 32
s = model_sizes[1]
yolo_model = YOLO(f'../../../storage/yolo/models/yolo11{s[0]}-cls.pt').cuda()
x = torch.randn(1, 3, dim, dim) / 255
x += abs(x.min())
x = x.to('cuda')

In [ ]:
_ = yolo_model(x)

In [ ]:
with OperationsCounterMode(yolo_model) as ops_counter:
    yolo_model(x, verbose=False)
s, dim, ops_counter.total_operations / 1e9

In [ ]:
p_medium = np.asarray([3.7003e-13, 1.4e-08, 4.8944])
np.polyval(p_medium, dim)

In [ ]:
s2results = dict()

for s in model_sizes:
    print(f'Model size: {s}')
    
    l = list()
    for dim in tqdm(dims, disable=True):
        yolo_model = YOLO(f'../../../storage/yolo/models/yolo11{s[0]}-cls.pt')
        x = torch.randn(1, 3, dim, dim) / 255
        x += abs(x.min())
        _ = yolo_model(x)
        with OperationsCounterMode(yolo_model) as ops_counter:
            yolo_model(x, verbose=False)
        
        l.append(ops_counter.total_operations / 1e9)  # in GigaOP(s)

    s2results[s] = l

In [ ]:
xticks = [dim**2 for dim in dims]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, (ax, s) in enumerate(zip(axes, s2results)):
    ax.plot(xticks, s2results[s], c=f'C{i}', linewidth=2)
    
    p = np.polyfit(xticks, s2results[s], deg=2)
    print(f'Size: {s}\tpolynomial = {str(p)}')
    y = [np.polyval(p, x) for x in xticks]
    ax.plot(xticks, y, c='black', linewidth=2, linestyle='--', label=str(p))
    
    ax.legend()
    ax.set_title(s)
    ax.set_xticklabels(dims)
    ax.set_xlabel('Figure dimension (pixel x pixel)')
    ax.set_ylabel('GigaOP(s)')

fig.tight_layout()
plt.show()
plt.close(fig)